<a href="https://colab.research.google.com/github/arashhadadex/Merge-DataFrames-in-Pandas/blob/main/pd_merge().ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Merge DataFrames in Pandas with pd.merge()**

In [ ]:
import pandas as pd
import sqlite3
import urllib.request

## **Loading the Chinook Dataset**
### database is freely available as a SQLite file.
### Every example uses the Chinook database, a realistic sample dataset modelling a digital music store with customers, invoices, tracks, artists, and employees across 11 related tables.

In [ ]:
# Download the Chinook SQLite database
url = "https://github.com/lerocha/chinook-database/raw/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite"
urllib.request.urlretrieve(url, "chinook.db")

# Connect and load the tables we need
conn = sqlite3.connect("chinook.db")

In [ ]:
customers  = pd.read_sql("SELECT * FROM Customer", conn)
invoices   = pd.read_sql("SELECT * FROM Invoice", conn)
inv_items  = pd.read_sql("SELECT * FROM InvoiceLine", conn)
tracks     = pd.read_sql("SELECT * FROM Track", conn)
artists    = pd.read_sql("SELECT * FROM Artist", conn)
albums     = pd.read_sql("SELECT * FROM Album", conn)
genres     = pd.read_sql("SELECT * FROM Genre", conn)
employees  = pd.read_sql("SELECT * FROM Employee", conn)

conn.close()

In [ ]:
print(f"Customers:     {customers.shape}")
print(f"Invoices:      {invoices.shape}")
print(f"Invoice lines: {inv_items.shape}")
print(f"Tracks:        {tracks.shape}")

Customers:     (59, 13)
Invoices:      (412, 9)
Invoice lines: (2240, 5)
Tracks:        (3503, 9)


In [ ]:
customers.head()

,CustomerId,FirstName,LastName,Company,Address,City,State,Country,PostalCode,Phone,Fax,Email,SupportRepId
0,1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,+55 (12) 3923-5555,+55 (12) 3923-5566,luisg@embraer.com.br,3
1,2,Leonie,Köhler,None,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,+49 0711 2842222,None,leonekohler@surfeu.de,5
2,3,François,Tremblay,None,1498 rue Bélanger,Montréal,QC,Canada,H2G 1A7,+1 (514) 721-4711,None,ftremblay@gmail.com,3
3,4,Bjørn,Hansen,None,Ullevålsveien 14,Oslo,None,Norway,0171,+47 22 44 22 22,None,bjorn.hansen@yahoo.no,4
4,5,František,Wichterlová,JetBrains s.r.o.,Klanova 9/506,Prague,None,Czech Republic,14700,+420 2 4172 5555,+420 2 4172 5555,frantisekw@jetbrains.com,4


In [ ]:
invoices.head()

,InvoiceId,CustomerId,InvoiceDate,BillingAddress,BillingCity,BillingState,BillingCountry,BillingPostalCode,Total
0,1,2,2021-01-01 00:00:00,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,1.98
1,2,4,2021-01-02 00:00:00,Ullevålsveien 14,Oslo,None,Norway,0171,3.96
2,3,8,2021-01-03 00:00:00,Grétrystraat 63,Brussels,None,Belgium,1000,5.94
3,4,14,2021-01-06 00:00:00,8210 111 ST NW,Edmonton,AB,Canada,T6G 2C7,8.91
4,5,23,2021-01-11 00:00:00,69 Salem Street,Boston,MA,USA,2113,13.86


In [ ]:
inv_items.head()

,InvoiceLineId,InvoiceId,TrackId,UnitPrice,Quantity
0,1,1,2,0.99,1
1,2,1,4,0.99,1
2,3,2,6,0.99,1
3,4,2,8,0.99,1
4,5,2,10,0.99,1


In [ ]:
tracks.head()

,TrackId,Name,AlbumId,MediaTypeId,GenreId,Composer,Milliseconds,Bytes,UnitPrice
0,1,For Those About To Rock (We Salute You),1,1,1,"Angus Young, Malcolm Young, Brian Johnson",343719,11170334,0.99
1,2,Balls to the Wall,2,2,1,"U. Dirkschneider, W. Hoffmann, H. Frank, P. Ba...",342562,5510424,0.99
2,3,Fast As a Shark,3,2,1,"F. Baltes, S. Kaufman, U. Dirkscneider & W. Ho...",230619,3990994,0.99
3,4,Restless and Wild,3,2,1,"F. Baltes, R.A. Smith-Diesel, S. Kaufman, U. D...",252051,4331779,0.99
4,5,Princess of the Dawn,3,2,1,Deaffy & R.A. Smith-Diesel,375418,6290521,0.99


In [ ]:
albums.head()

,AlbumId,Title,ArtistId
0,1,For Those About To Rock We Salute You,1
1,2,Balls to the Wall,2
2,3,Restless and Wild,2
3,4,Let There Be Rock,1
4,5,Big Ones,3


In [ ]:
genres.head()

,GenreId,Name
0,1,Rock
1,2,Jazz
2,3,Metal
3,4,Alternative & Punk
4,5,Rock And Roll


# **What Is pd.merge()?**

## pd.merge() combines two DataFrames horizontally by aligning rows on one or more shared columns; the same operation SQL calls a JOIN.
## The four join types work similarly to SQL:
### **Inner join:** only rows with matching keys in both DataFrames
### **Left join:** all rows from the left, matched rows from the right (NaN where no match)
### **Right join:** all rows from the right, matched rows from the left (NaN where no match)
### **Outer join:** all rows from both, with NaN where there is no match on either side

### **pd.merge(left, right, how='inner', on='column_name')**

# **1. Inner Join: Only Matching Records**

### The inner join is the default. It returns rows where the key exists in both ### DataFrames, and unmatched rows from either side are dropped.
### **Use case:** Find all customers who have placed at least one order.

In [ ]:
# Merge customers with their invoices
customers_with_orders = pd.merge(
    customers[['CustomerId', 'FirstName', 'LastName', 'Country']],
    invoices[['InvoiceId', 'CustomerId', 'InvoiceDate', 'Total']],
    on='CustomerId',
    how='inner'
)

print(f"Shape: {customers_with_orders.shape}")
print(customers_with_orders.head())

Shape: (412, 7)
   CustomerId FirstName   LastName Country  InvoiceId          InvoiceDate  \
0           1      Luís  Gonçalves  Brazil         98  2022-03-11 00:00:00   
1           1      Luís  Gonçalves  Brazil        121  2022-06-13 00:00:00   
2           1      Luís  Gonçalves  Brazil        143  2022-09-15 00:00:00   
3           1      Luís  Gonçalves  Brazil        195  2023-05-06 00:00:00   
4           1      Luís  Gonçalves  Brazil        316  2024-10-27 00:00:00   

   Total  
0   3.98  
1   3.96  
2   5.94  
3   0.99  
4   1.98  


### The result has 412 rows, one per invoice. Customer 1 (Luís) appears multiple times because he placed multiple orders. This is expected and correct for a one-to-many relationship.

In [ ]:
print(customers_with_orders['CustomerId'].nunique())
# 59 — all customers have at least one invoice in this dataset

59


# **2. Left Join: Keep All Left Records**

### The left join keeps every row from the left DataFrame, no matter of whether a match exists in the right. Where there is no match, right-side columns are filled with NaN.
### **Use case:** Show all customers and their total spending, which includes customers who have never ordered.

In [ ]:
# Aggregate total spend per customer
total_spend = invoices.groupby('CustomerId', as_index=False)['Total'].sum()
total_spend.columns = ['CustomerId', 'TotalSpend']

# Left join: keep all customers, even those with no orders
all_customers = pd.merge(
    customers[['CustomerId', 'FirstName', 'LastName', 'Country']],
    total_spend,
    on='CustomerId',
    how='left'
)

print(f"All customers: {len(all_customers)}")
print(f"Customers with NaN spend: {all_customers['TotalSpend'].isna().sum()}")
print(all_customers.sort_values('TotalSpend', ascending=False).head())

All customers: 59
Customers with NaN spend: 0
    CustomerId FirstName    LastName         Country  TotalSpend
5            6    Helena        Holý  Czech Republic       49.62
25          26   Richard  Cunningham             USA       47.62
56          57      Luis       Rojas           Chile       46.62
44          45  Ladislav      Kovács         Hungary       45.62
45          46      Hugh    O'Reilly         Ireland       45.62


### In this dataset, every customer has at least one invoice, so no NaN values appear. In a real production database, new customers with no purchases would show NaN in the TotalSpend feature; the left join ensures they still exist in the output rather than being dropped by an inner join.
### **Real-world importance of left join:** If you use an inner join for this query and your manager asks "how many customers have we acquired but not yet converted?", you will incorrectly report zero because the inner join silently dropped them.

# **3. Right Join: Keep All Right Records**
### The right join is the mirror of the left join - every row from the right DataFrame is kept, with NaN for unmatched left-side columns.
### **Use case:** Show all invoices and their associated customer details, while keeping all invoices even if customer data is missing (for example, after a data migration where some customer records were lost).

In [ ]:
merged_right = pd.merge(
    customers[['CustomerId', 'FirstName', 'LastName', 'Country']],
    invoices[['InvoiceId', 'CustomerId', 'InvoiceDate', 'Total']],
    on='CustomerId',
    how='right'
)

print(f"Shape: {merged_right.shape}")
print(f"Invoices with missing customer data: {merged_right['FirstName'].isna().sum()}")

Shape: (412, 7)
Invoices with missing customer data: 0


### **Practical note:** In practice, right joins are rare. Any right join can be rewritten as a left join by swapping the DataFrame order, and left joins are more readable because the "primary" table is always on the left. Many experienced analysts never use the right join at all.

# 4. **Outer Join: All Records from Both Sides**

### The outer join (also called a full outer join) returns every row from both DataFrames. Where no match exists on either side, NaN fills the gaps.
### **Use case:** Audit which tracks have never appeared on any invoice, and which invoice lines reference tracks that no longer exist in the catalog.

In [ ]:
# Merge tracks with invoice line items
track_sales = pd.merge(
    tracks[['TrackId', 'Name', 'GenreId', 'UnitPrice']],
    inv_items[['InvoiceLineId', 'TrackId', 'Quantity']],
    on='TrackId',
    how='outer'
)

never_sold = track_sales[track_sales['InvoiceLineId'].isna()]
ghost_sales = track_sales[track_sales['Name'].isna()]

print(f"Total tracks:          {len(tracks)}")
print(f"Tracks never sold:     {len(never_sold)}")
print(f"Sales with no track:   {len(ghost_sales)}")
print(never_sold[['TrackId', 'Name', 'UnitPrice']].head())

Total tracks:          3503
Tracks never sold:     1519
Sales with no track:   0
    TrackId               Name  UnitPrice
7         7    Let's Get It Up       0.99
13       11             C.O.D.       0.99
19       17  Let There Be Rock       0.99
20       18     Bad Boy Boogie       0.99
25       22  Whole Lotta Rosie       0.99


### Over 1,500 tracks in the catalog have never been purchased - actionable business insight surfaced by an outer join that an inner join would have completely hidden.

# **5. Merging on Differently Named Columns**

### The on parameter only works when the key column has the same name in both DataFrames. When names differ, use left_onand right_on.
### **Use case:** Join albums to artists - the key is ArtistId in both tables, but let's demonstrate with a renamed column to illustrate the pattern.

In [ ]:
# Simulate a scenario where the key has different names
artists_renamed = artists.rename(columns={'ArtistId': 'artist_id'})

# Now 'ArtistId' exists in albums, 'artist_id' exists in artists_renamed
albums_with_artists = pd.merge(
    albums[['AlbumId', 'Title', 'ArtistId']],
    artists_renamed[['artist_id', 'Name']],
    left_on='ArtistId',
    right_on='artist_id',
    how='inner'
)

print(albums_with_artists.head())
print(f"\nColumns: {list(albums_with_artists.columns)}")

   AlbumId                                  Title  ArtistId  artist_id  \
0        1  For Those About To Rock We Salute You         1          1   
1        2                      Balls to the Wall         2          2   
2        3                      Restless and Wild         2          2   
3        4                      Let There Be Rock         1          1   
4        5                               Big Ones         3          3   

        Name  
0      AC/DC  
1     Accept  
2     Accept  
3      AC/DC  
4  Aerosmith  

Columns: ['AlbumId', 'Title', 'ArtistId', 'artist_id', 'Name']


## Notice that both ArtistId and artist_id appear in the result. Pandas keeps both key columns when you use left_on/right_on. Drop the redundant one:

In [ ]:
albums_with_artists = albums_with_artists.drop(columns=['artist_id'])
print(albums_with_artists)

     AlbumId                                              Title  ArtistId  \
0          1              For Those About To Rock We Salute You         1   
1          2                                  Balls to the Wall         2   
2          3                                  Restless and Wild         2   
3          4                                  Let There Be Rock         1   
4          5                                           Big Ones         3   
..       ...                                                ...       ...   
342      343                             Respighi:Pines of Rome       226   
343      344  Schubert: The Late String Quartets & String Qu...       272   
344      345                                Monteverdi: L'Orfeo       273   
345      346                              Mozart: Chamber Music       274   
346      347  Koyaanisqatsi (Soundtrack from the Motion Pict...       275   

                                                  Name  
0                 

# **6. Handling Duplicate Column Names with suffixes**

### When both DataFrames share column names beyond the key, Pandas appends **_x** and **_y** by default to distinguish them. You can control this with the suffixes parameter.
### **Use case:** Join invoice lines to tracks, both have a UnitPrice column.

In [ ]:
# Default behavior — suffixes _x and _y
merged_default = pd.merge(
    inv_items[['InvoiceLineId', 'TrackId', 'UnitPrice', 'Quantity']],
    tracks[['TrackId', 'Name', 'UnitPrice']],
    on='TrackId',
    how='inner'
)
print(merged_default.columns.tolist())
# ['InvoiceLineId', 'TrackId', 'UnitPrice_x', 'Quantity', 'Name', 'UnitPrice_y']

# Custom suffixes — much more readable
merged_custom = pd.merge(
    inv_items[['InvoiceLineId', 'TrackId', 'UnitPrice', 'Quantity']],
    tracks[['TrackId', 'Name', 'UnitPrice']],
    on='TrackId',
    how='inner',
    suffixes=('_sold', '_catalog')
)
print(merged_custom.columns.tolist())
# ['InvoiceLineId', 'TrackId', 'UnitPrice_sold', 'Quantity', 'Name', 'UnitPrice_catalog']

# Check if any tracks were sold at a different price than the catalog
price_discrepancy = merged_custom[
    merged_custom['UnitPrice_sold'] != merged_custom['UnitPrice_catalog']
]
print(f"Price discrepancies: {len(price_discrepancy)}")

['InvoiceLineId', 'TrackId', 'UnitPrice_x', 'Quantity', 'Name', 'UnitPrice_y']
['InvoiceLineId', 'TrackId', 'UnitPrice_sold', 'Quantity', 'Name', 'UnitPrice_catalog']
Price discrepancies: 0


### Always use descriptive suffixes in production code. **_x** and **_y** become confusing within days, especially when you revisit a notebook you wrote months ago.

# **7. Multi-Key Merges**

### Sometimes, a single column is not unique enough to identify a row; you need to merge on multiple columns simultaneously.
### **Use case:** Build a complete sales report joining invoice lines to tracks, albums, and genres in one pipeline.

In [ ]:
print(tracks_full.columns)

Index(['TrackId', 'Name_track', 'AlbumId', 'GenreId', 'UnitPrice',
       'Name_genre', 'Title', 'ArtistId', 'Name'],
      dtype='object')


In [ ]:
tracks_full['Name'] = 'Name_artist' # Fasle

In [ ]:
tracks_full = tracks_full.rename(columns={'Name': 'Name_artist'})

# Or

# tracks_full.rename(columns={'Name': 'Name_artist'}, inplace=True)

In [ ]:
tracks_full

,TrackId,Name_track,AlbumId,GenreId,UnitPrice,Name_genre,Title,ArtistId,Name
0,1,For Those About To Rock (We Salute You),1,1,0.99,Rock,For Those About To Rock We Salute You,1,AC/DC
1,2,Balls to the Wall,2,1,0.99,Rock,Balls to the Wall,2,Accept
2,3,Fast As a Shark,3,1,0.99,Rock,Restless and Wild,2,Accept
3,4,Restless and Wild,3,1,0.99,Rock,Restless and Wild,2,Accept
4,5,Princess of the Dawn,3,1,0.99,Rock,Restless and Wild,2,Accept
...,...,...,...,...,...,...,...,...,...
3498,3499,Pini Di Roma (Pinien Von Rom) \ I Pini Della V...,343,24,0.99,Classical,Respighi:Pines of Rome,226,Eugene Ormandy
3499,3500,"String Quartet No. 12 in C Minor, D. 703 ""Quar...",344,24,0.99,Classical,Schubert: The Late String Quartets & String Qu...,272,Emerson String Quartet
3500,3501,"L'orfeo, Act 3, Sinfonia (Orchestra)",345,24,0.99,Classical,Monteverdi: L'Orfeo,273,"C. Monteverdi, Nigel Rogers - Chiaroscuro; Lon..."
3501,3502,"Quintet for Horn, Violin, 2 Violas, and Cello ...",346,24,0.99,Classical,Mozart: Chamber Music,274,Nash Ensemble


In [ ]:
# Step 1: tracks with genre names
tracks_with_genre = pd.merge(
    tracks[['TrackId', 'Name', 'AlbumId', 'GenreId', 'UnitPrice']],
    genres[['GenreId', 'Name']],
    on='GenreId',
    how='left',
    suffixes=('_track', '_genre')
)

# Step 2: add album and artist info
tracks_full = pd.merge(
    tracks_with_genre,
    albums[['AlbumId', 'Title', 'ArtistId']],
    on='AlbumId',
    how='left'
)

tracks_full = pd.merge(
    tracks_full,
    artists[['ArtistId', 'Name']],
    on='ArtistId',
    how='left',
    suffixes=('', '_artist')
)

tracks_full = tracks_full.rename(columns={'Name': 'Name_artist'})

# Step 3: join to invoice lines
sales_full = pd.merge(
    inv_items[['InvoiceLineId', 'InvoiceId', 'TrackId', 'Quantity']],
    tracks_full[['TrackId', 'Name_track', 'Name_genre', 'Title', 'Name_artist', 'UnitPrice']],
    on='TrackId',
    how='inner'
)

# Step 4: join the invoices for the date and the customer
sales_full = pd.merge(
    sales_full,
    invoices[['InvoiceId', 'CustomerId', 'InvoiceDate', 'BillingCountry']],
    on='InvoiceId',
    how='inner'
)

print(f"Complete sales records: {sales_full.shape}")
print(sales_full[['Name_track', 'Name_genre', 'Name_artist', 'Quantity', 'UnitPrice', 'BillingCountry']].head())

Complete sales records: (2240, 12)
              Name_track Name_genre Name_artist  Quantity  UnitPrice  \
0      Balls to the Wall       Rock      Accept         1       0.99   
1      Restless and Wild       Rock      Accept         1       0.99   
2  Put The Finger On You       Rock       AC/DC         1       0.99   
3       Inject The Venom       Rock       AC/DC         1       0.99   
4             Evil Walks       Rock       AC/DC         1       0.99   

  BillingCountry  
0        Germany  
1        Germany  
2         Norway  
3         Norway  
4         Norway  


## Now answer the original question - which customers spent the most on rock music?

In [ ]:
rock_sales = sales_full[sales_full['Name_genre'] == 'Rock'].copy()
rock_sales['Revenue'] = rock_sales['Quantity'] * rock_sales['UnitPrice']

top_rock_customers = pd.merge(
    rock_sales.groupby('CustomerId', as_index=False)['Revenue'].sum(),
    customers[['CustomerId', 'FirstName', 'LastName', 'Country']],
    on='CustomerId',
    how='inner'
).sort_values('Revenue', ascending=False).head(10)

print(top_rock_customers)

    CustomerId  Revenue  FirstName  LastName    Country
9           10    28.71    Eduardo   Martins     Brazil
28          29    24.75     Robert     Brown     Canada
54          55    21.78       Mark    Taylor  Australia
49          50    21.78    Enrique     Muñoz      Spain
48          49    21.78  Stanisław    Wójcik     Poland
7            8    20.79       Daan   Peeters    Belgium
37          38    20.79     Niklas  Schröder    Germany
8            9    20.79       Kara   Nielsen    Denmark
17          18    18.81   Michelle    Brooks        USA
32          33    18.81      Ellie  Sullivan     Canada


# **8. Validating Merge Results**

### A merge that runs without errors can still be wrong. Always validate.

In [ ]:
# Check 1: row count sanity
before = len(customers)
after = len(pd.merge(customers, invoices, on='CustomerId', how='inner'))
print(f"Customers before: {before}, rows after inner join: {after}")
# If after > before, it's a one-to-many — expected
# If after < before, rows were dropped — investigate

# Check 2: NaN audit after left join
result = pd.merge(customers, invoices, on='CustomerId', how='left')
nan_counts = result.isna().sum()
print(nan_counts[nan_counts > 0])

# Check 3: use validate to catch unexpected duplicates
try:
    pd.merge(
        customers,
        invoices,
        on='CustomerId',
        how='inner',
        validate='1:1'   # expects each CustomerId to appear once in both
    )
except pd.errors.MergeError as e:
    print(f"Validation failed: {e}")
    # MergeError: Merge keys are not unique in the right dataset
    # This confirms it's a 1:many relationship — use validate='1:m' instead

# Correct validation for this relationship
pd.merge(customers, invoices, on='CustomerId', how='inner', validate='1:m')
print("Validation passed: confirmed 1-to-many relationship")

Customers before: 59, rows after inner join: 412
Company              342
State                202
PostalCode            28
Phone                  7
Fax                  328
BillingState         202
BillingPostalCode     28
dtype: int64
Validation failed: Merge keys are not unique in right dataset; not a one-to-one merge
Validation passed: confirmed 1-to-many relationship


### The validate parameter is one of the most underused features in pd.merge(). It raises a MergeError immediately if your assumption about key uniqueness is wrong, catching data quality issues before they silently corrupt downstream analysis.

# **9. Performance Tips for Large DataFrames**

### pd.merge() performance degrades quickly on large datasets if you are not careful.

## **Tip 1: Select only the columns you need before merging**
### Reducing column count before merging cuts both memory usage and processing time significantly.

In [ ]:
# Slow — merges all 13 columns from customers
result = pd.merge(customers, invoices, on='CustomerId')

# Fast — merges only what you need
result = pd.merge(
    customers[['CustomerId', 'FirstName', 'LastName', 'Country']],
    invoices[['CustomerId', 'InvoiceId', 'Total', 'InvoiceDate']],
    on='CustomerId'
)

## **Tip 2: Use categorical dtype for string key columns**

In [ ]:
# Convert string keys to categorical before merging
customers['Country'] = customers['Country'].astype('category')
invoices['BillingCountry'] = invoices['BillingCountry'].astype('category')

# Categorical merges use integer comparisons internally — much faster

## **Tip 3: Set the index on the key column for repeated merges**

In [ ]:
customers_indexed = customers.set_index('CustomerId')
invoices_indexed = invoices.set_index('CustomerId')

# Use DataFrame.join() for index-based merging — faster than pd.merge() on large data
result = customers_indexed.join(invoices_indexed, how='inner', lsuffix='_cust', rsuffix='_inv')

## **Tip 4: Check for duplicates in key columns before merging**

In [ ]:
# Unexpected duplicates in key columns cause row explosion
print(f"Duplicate CustomerIds in customers: {customers['CustomerId'].duplicated().sum()}")
print(f"Duplicate CustomerIds in invoices: {invoices['CustomerId'].duplicated().sum()}")
# customers: 0 (good — unique key)
# invoices: 353 (expected — one customer, many invoices)

Duplicate CustomerIds in customers: 0
Duplicate CustomerIds in invoices: 353


### If you expect a 1:1 merge and your left key has duplicates you did not anticipate, the result will silently have far more rows than either input DataFrame.

# **10. pd.merge() vs pd.concat() vs DataFrame.join()**
### These three functions are frequently confused

In [ ]:
# pd.concat() — stacking rows (like SQL UNION)
q1_sales = invoices[invoices['InvoiceDate'].str.startswith('2021-01')]
q2_sales = invoices[invoices['InvoiceDate'].str.startswith('2021-04')]
combined = pd.concat([q1_sales, q2_sales], ignore_index=True)
print(f"Q1: {len(q1_sales)}, Q2: {len(q2_sales)}, Combined: {len(combined)}")

# DataFrame.join() — index-based merge
customers_idx = customers[['CustomerId', 'Country']].set_index('CustomerId')
invoices_idx = invoices[['CustomerId', 'Total']].set_index('CustomerId')
joined = customers_idx.join(invoices_idx, how='inner')
print(joined.head())

Q1: 6, Q2: 7, Combined: 13
           Country  Total
CustomerId               
1           Brazil   3.98
1           Brazil   3.96
1           Brazil   5.94
1           Brazil   0.99
1           Brazil   1.98


### Use pd.merge() for relational joins on column values. Use pd.concat() for appending rows. Use DataFrame.join() when your key is already the index, and you want slightly cleaner syntax.

# **Real-World Summary: The Complete Analysis Pipeline**

### Putting it all together, a full analysis pipeline answering: what are the top 5 genres by revenue, and which countries drive that revenue?

In [ ]:
# Build the complete picture
pipeline = (
    inv_items[['TrackId', 'Quantity', 'UnitPrice']]
    .assign(Revenue=lambda x: x['Quantity'] * x['UnitPrice'])
    .pipe(lambda df: pd.merge(df, tracks[['TrackId', 'GenreId']], on='TrackId', how='left'))
    .pipe(lambda df: pd.merge(df, genres[['GenreId', 'Name']], on='GenreId', how='left'))
    .pipe(lambda df: pd.merge(df, inv_items[['InvoiceLineId', 'InvoiceId']].drop_duplicates(),
                               left_index=True, right_index=True, how='left'))
)

# Top genres by total revenue
genre_revenue = (
    pd.merge(
        inv_items[['TrackId', 'Quantity', 'UnitPrice']].assign(
            Revenue=lambda x: x['Quantity'] * x['UnitPrice']
        ),
        tracks[['TrackId', 'GenreId']],
        on='TrackId', how='left'
    )
    .pipe(lambda df: pd.merge(df, genres[['GenreId', 'Name']], on='GenreId', how='left'))
    .groupby('Name')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)
genre_revenue.columns = ['Genre', 'Total Revenue']
genre_revenue['Total Revenue'] = genre_revenue['Total Revenue'].map('${:,.2f}'.format)
print(genre_revenue.to_string(index=False))

             Genre Total Revenue
              Rock       $826.65
             Latin       $382.14
             Metal       $261.36
Alternative & Punk       $241.56
          TV Shows        $93.53
